# Image → caption / subject explanation: 20 words, filtered tokens and patch interventions (v2)

**Kaggle × 2 GPUs, adapted from the uploaded `analysis-concept-grad-heatmap-v1(1).ipynb`.** Dataset: [zintom/blackbox-adversarial-attack-dataset](https://www.kaggle.com/datasets/zintom/blackbox-adversarial-attack-dataset).

1. Generate a **20-word English caption** (deterministically check the word count); retain dataset caption separately if present. Identify the **grammatical main subject** of the caption (not a verified object label).
2. **Select content words using spaCy POS + dependency parsing:** nouns, proper nouns, descriptive adjectives, main verbs, numbers; exclude pronouns, determiners, auxiliaries and other function classes. Preserve exact character spans and report the choices. No manually curated stopword prompts. Select at most `MAX_CONTENT_TOKENS`.
3. Produce **two visible heatmap galleries** (full caption and individually selected words) using exactly the same expression for each model: **Qwen3-VL query-token → image-token attention**, **official CLIP Surgery**, **Grad-ECLIP author's last-layer single-head gradient formulation** (not generic Grad-CAM).
4. Quantify word/patch sensitivity on **vanilla CLIP ViT-B/16** and **Qwen3-VL target teacher-forced likelihood** separately. Run CLIP grid deletion on every patch; run Qwen patch deletion on a **clearly marked sampled subset** to save GPU time. Rank per-token Qwen image-vs-blank log-probability difference and CLIP caption word LOO. Record the strongest CLIP patches for *subject* and *caption*.
5. Show original images and figures **inline**; save heatmaps, per-token scores, selected-word × image-patch interaction PNG/CSV, per-patch CSV/NPZ and metadata to `/kaggle/working/concept_grad_heatmap_v2/`.

**Interpretation and limitations.** Raw attention is *not* causal influence. CLIP word removal can change grammatical meaning; image deletion can introduce out-of-distribution artifacts. Grad-ECLIP's last-layer single-head reconstructed score is *not* identical to standard multihead CLIP. Likelihood results refer to a **fixed target string scored under teacher forcing**, not a re-generated caption/subject after ablation. Qwen's subject target is a parse of the candidate caption and may not match the independently generated answer.

**Method references:** [CLIP Surgery (Pattern Recognition 2025)](https://doi.org/10.1016/j.patcog.2025.111409) / [official code](https://github.com/xmed-lab/CLIP_Surgery); [Grad-ECLIP (ICML 2024)](https://proceedings.mlr.press/v235/zhao24p.html) / [author code](https://github.com/Cyang-Zhao/Grad-Eclip); [Qwen3-VL Transformers implementation](https://huggingface.co/docs/transformers/model_doc/qwen3_vl); [spaCy linguistic features](https://spacy.io/usage/linguistic-features).

**Run:** Kaggle accelerator `GPU T4 ×2`, Internet ON for model downloads; attach dataset. Start with `MAX_IMAGES=2`. The notebook shows samples before model downloads. There are no precomputed experimental results.

In [ ]:
# Kaggle Internet ON for first run. For torch on Kaggle use its preinstalled CUDA build.
%pip -q install -U "transformers>=4.57.1,<5" "accelerate>=1.6" "ftfy" "regex" "kagglehub" "nbformat" "pandas" "matplotlib" "opencv-python-headless" "safetensors" "setuptools<81" "spacy>=3.8,<4"
%pip -q install git+https://github.com/openai/CLIP.git
# English POS + dependency model, required for reproducible lexical filtering.
!python -m spacy download en_core_web_sm

In [ ]:
import os, sys, gc, re, json, math, time, copy, random, subprocess, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn.functional as F
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, InterpolationMode

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
if torch.cuda.is_available():
    torch.cuda.empty_cache()
assert torch.cuda.device_count() >= 2, 'This notebook needs 2 accessible GPUs. Set Kaggle Accelerator to GPU T4 x2.'
QWEN_DEVICE = torch.device('cuda:0')
CLIP_DEVICE = torch.device('cuda:1')
OUTPUT_DIR = Path('/kaggle/working/concept_grad_heatmap_v2' if Path('/kaggle/working').exists() else '/mnt/data/concept_grad_heatmap_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Experiment controls
MAX_IMAGES = 2                    # Smoke test first; set to None for full dataset
MAX_CONTENT_TOKENS = 6            # POS/dependency-filtered words per image; full caption always included
MAX_QWEN_TOKENS = 100             # for caption generation only
QWEN_MAX_PIXELS = 448 * 448      # reduce to 384**2 if T4 OOM; processor rounds to its patch constraints
QWEN_MIN_PIXELS = 256 * 256
QWEN_LAYER_FRACTIONS = (0.30, 0.55, 0.80, 0.96)
CLIP_SIDE = 224                 # official ViT-B/16 benchmark setup; 14x14 patch grid
TOP_MASK_FRACTION = 0.20       # deletion-test important-area fraction
RUN_DELETION = True
VISUALIZE_CAPTION_SUBTOKENS = True
SAVE_NPZ = True
CAPTION_WORDS = 20                 # exact regex word count, never silently accept fewer
CAPTION_GENERATION_ATTEMPTS = 3
PATCH_GRID = 7                     # full 7x7 CLIP occlusion grid (49 patches)
QWEN_PATCH_BUDGET = 8              # Qwen scores only *selected* patches; not all 49
PATCH_BLUR_KERNEL = 31
CLIP_PATCH_BATCH = 12
SHOW_DATASET_PREVIEW = 12
RUN_QWEN_PATCH_TEST = True
QWEN_MODEL_ID = 'Qwen/Qwen3-VL-2B-Instruct'
DATASET_SLUG = 'zintom/blackbox-adversarial-attack-dataset'
DATASET_DIR = None             # set e.g. '/kaggle/input/blackbox-adversarial-attack-dataset'
CAPTION_OVERRIDE = None        # optional caption applied to every image (usually leave None)

print('GPUs:', [(i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())])
print('PyTorch:', torch.__version__, '| outputs:', OUTPUT_DIR)

## 1. Discover image files and optional captions

The Kaggle dataset webpage does not expose a reliable machine-readable file listing to this notebook author. **No folder names or annotation columns are assumed.** This cell prints the exact mounted structure and attempts to resolve JSON/JSONL/CSV/Parquet caption records, including COCO-style `images`/`annotations`. Check `preview` before running inference. It does not silently substitute another dataset.

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
TEXT_EXTS = {'.csv', '.json', '.jsonl', '.parquet'}
CAPTION_KEYS = ('caption', 'captions', 'description', 'text', 'sentence')
IMAGE_KEYS = ('file_name', 'filename', 'image_path', 'path', 'image', 'img', 'file')
ID_KEYS = ('image_id', 'id')

def discover_root():
    if DATASET_DIR is not None:
        root = Path(DATASET_DIR)
        if not root.exists(): raise FileNotFoundError(root)
        return root
    base = Path('/kaggle/input')
    dirs = [p for p in base.glob('*') if p.is_dir()] if base.exists() else []
    exact = next((p for p in dirs if p.name == 'blackbox-adversarial-attack-dataset'), None)
    if exact: return exact
    matches = [p for p in dirs if 'blackbox' in p.name.lower() and 'attack' in p.name.lower()]
    if len(matches) == 1: return matches[0]
    print('Mounted Kaggle input folders:', [p.name for p in dirs])
    import kagglehub
    root = Path(kagglehub.dataset_download(DATASET_SLUG))
    return root

ROOT = discover_root()
print('DATASET ROOT:', ROOT)
files = sorted((p for p in ROOT.rglob('*') if p.is_file()), key=lambda p: str(p))
print('Total files:', len(files))
print('First files:', '\n'.join('  ' + str(p.relative_to(ROOT)) for p in files[:30]))
image_files = [p for p in files if p.suffix.lower() in IMAGE_EXTS]
meta_files = [p for p in files if p.suffix.lower() in TEXT_EXTS]
print('Images:', len(image_files), '| metadata candidates:', len(meta_files))
assert image_files, 'No ordinary image files found. Inspect archive formats / set DATASET_DIR appropriately.'

# Map by basename/stem/relative path, resolving ambiguous matches conservatively.
from collections import defaultdict
lookup = defaultdict(list)
for p in image_files:
    for k in {p.name.lower(), p.stem.lower(), p.relative_to(ROOT).as_posix().lower()}:
        lookup[k].append(p)

def resolve_image(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).replace('\\', '/').strip().lower().lstrip('./')
    candidates = lookup.get(s, []) or lookup.get(Path(s).name, []) or lookup.get(Path(s).stem, [])
    return str(candidates[0]) if len(candidates) == 1 else None

def get_first(rec, keys):
    for k in keys:
        if k in rec and rec[k] is not None and not isinstance(rec[k], (list, dict)):
            return rec[k]
    return None

def records_from_metadata(path):
    if path.stat().st_size > 120 * 1024 * 1024: return []
    try:
        if path.suffix.lower() == '.csv':
            return pd.read_csv(path, nrows=100000).to_dict('records')
        if path.suffix.lower() == '.parquet':
            return pd.read_parquet(path).head(100000).to_dict('records')
        if path.suffix.lower() == '.jsonl':
            with path.open(encoding='utf-8') as f: return [json.loads(line) for _, line in zip(range(100000), f) if line.strip()]
        obj = json.loads(path.read_text(encoding='utf-8'))
        if isinstance(obj, list): return obj
        if isinstance(obj, dict):
            if 'annotations' in obj and 'images' in obj:
                names = {str(r.get('id')): get_first(r, IMAGE_KEYS) for r in obj['images'] if isinstance(r, dict)}
                return [dict(image=names.get(str(a.get('image_id'))), caption=get_first(a, CAPTION_KEYS))
                        for a in obj['annotations'] if isinstance(a, dict)]
            for key in ('data', 'records', 'items', 'samples'):
                if isinstance(obj.get(key), list): return obj[key]
    except Exception as e:
        print('Skipping metadata', path.name, type(e).__name__, str(e)[:100])
    return []

caption_map = {}
for mf in meta_files[:80]:
    for r in records_from_metadata(mf):
        if not isinstance(r, dict): continue
        cap = get_first(r, CAPTION_KEYS)
        direct = resolve_image(get_first(r, IMAGE_KEYS))
        if direct is None and get_first(r, ID_KEYS) is not None:
            direct = resolve_image(str(get_first(r, ID_KEYS)))
        if direct and isinstance(cap, str) and cap.strip() and direct not in caption_map:
            caption_map[direct] = cap.strip()
print('Images with matched captions:', len(caption_map))

samples = [{'image_id': p.stem, 'image_path': str(p),
            'reference_caption': CAPTION_OVERRIDE or caption_map.get(str(p))}
           for p in image_files]
if MAX_IMAGES is not None:
    samples = samples[:MAX_IMAGES]
preview = pd.DataFrame(samples)
display(preview.head(10))
print('Selected:', len(samples))

## 1b. Inspect actual dataset images **before** loading models

The next cell renders the actual mounted samples and reference captions, allowing you to check whether discovery matched the intended dataset. Set `SHOW_DATASET_PREVIEW` to control the contact sheet. This is not a generated/synthetic substitute for dataset images.

In [ ]:
from IPython.display import display
nprev = min(len(samples), SHOW_DATASET_PREVIEW)
fig, axs = plt.subplots(math.ceil(nprev/4), 4, figsize=(15, 3.5*math.ceil(nprev/4)), squeeze=False)
for ax in axs.flat: ax.axis('off')
for i in range(nprev):
    s = samples[i]
    ax = axs.flat[i]
    ax.imshow(load_image_preview := ImageOps.exif_transpose(Image.open(s['image_path'])).convert('RGB'))
    ax.set_title(str(s['image_id'])[:30], fontsize=9)
    if s.get('reference_caption'):
        ax.set_xlabel(str(s['reference_caption'])[:80], fontsize=7)
    ax.axis('off')
fig.suptitle('Original samples from the mounted Kaggle dataset (before inference)')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'dataset_preview.png', dpi=130, bbox_inches='tight')
plt.show(); plt.close(fig)

## 2. Load Qwen on GPU 0 and two official CLIP models on GPU 1

The Surgery repository exports its own `clip` package. Import it under a distinct package name so it cannot shadow the stock OpenAI `clip` used by Grad-ECLIP. The Surgery checkpoint's architecture change is invoked via `CS-ViT-B/16`, *not* reproduced by a generic CLIP cosine map.

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, __version__ as HF_VERSION
import clip as openai_clip
import importlib.util

SURGERY_SRC = OUTPUT_DIR / '_CLIP_Surgery'
if not (SURGERY_SRC / 'clip' / '__init__.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/xmed-lab/CLIP_Surgery.git', str(SURGERY_SRC)], check=True)
spec = importlib.util.spec_from_file_location('surgery_clip', SURGERY_SRC / 'clip' / '__init__.py',
                                              submodule_search_locations=[str(SURGERY_SRC / 'clip')])
surgery_clip = importlib.util.module_from_spec(spec)
sys.modules['surgery_clip'] = surgery_clip
spec.loader.exec_module(surgery_clip)

QWEN_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print('transformers:', HF_VERSION, '| Qwen dtype:', QWEN_DTYPE)
qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
# Official Qwen3-VL pixel budget: size longest_edge=max_pixels, shortest_edge=min_pixels.
qwen_processor.image_processor.size = {'longest_edge': QWEN_MAX_PIXELS, 'shortest_edge': QWEN_MIN_PIXELS}
qwen = Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID, torch_dtype=QWEN_DTYPE, attn_implementation='sdpa',
    low_cpu_mem_usage=True,
).to(QWEN_DEVICE).eval()
for p in qwen.parameters(): p.requires_grad_(False)

surgery_model, _ = surgery_clip.load('CS-ViT-B/16', device=str(CLIP_DEVICE), jit=False)
surgery_model.eval()
for p in surgery_model.parameters(): p.requires_grad_(False)
clip_model, _ = openai_clip.load('ViT-B/16', device=str(CLIP_DEVICE), jit=False)
clip_model.eval()
for p in clip_model.parameters(): p.requires_grad_(False)
print('Models ready | qwen GPU:', next(qwen.parameters()).device,
      '| surgery GPU:', next(surgery_model.parameters()).device,
      '| vanilla CLIP GPU:', next(clip_model.parameters()).device)

## 3. Generate a 20-word caption and select **real words from that caption** (no concept-generation prompt)

Qwen3-VL makes one descriptive caption. We validate that it contains exactly `CAPTION_WORDS` alphabetic/numeric words. If the answer has more than 20 words, truncate to the twentieth complete word (log `truncated=True`); if shorter, retry up to 3 times, then fail visibly rather than add invented tokens. This keeps raw responses for inspection. To avoid circular ground truth, the dataset's caption is saved as `reference_caption` but is **not** passed into this generation prompt.

Next, a frozen **spaCy English dependency parser** selects semantically substantive surface words (`NOUN`, `PROPN`, `ADJ`, finite/nonauxiliary `VERB`, `NUM`), exact caption offsets and lemmas. Subject is a noun chunk attached to a syntactic `nsubj` / `nsubjpass` (or first noun chunk fallback). These are **linguistic heuristics**, not verified visual grounding nor a learned saliency model. All selected words and reasons are shown in a table.

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm', disable=['ner'])
WORD_RE = re.compile(r"\b[\w]+(?:[-’'][\w]+)*\b", flags=re.UNICODE)
CAPTION_PROMPT = ('Describe the visible image in one concrete English sentence of exactly 20 words. '
                  'State the main subject and visually supported objects, attributes and actions. '
                  'Avoid speculation. Output only the sentence, without a word-count label.')
SUBJECT_SCORE_PROMPT = ('Identify the main visible subject in the image. Respond with the short subject noun phrase only.')

# Avoid passing any JSON-only system prompt to the likelihood scorer.
def qwen_inputs(image, text, system_text=None):
    messages = []
    if system_text:
        messages.append({'role':'system', 'content':[{'type':'text','text':system_text}]})
    messages.append({'role':'user', 'content':[{'type':'image'}, {'type':'text','text':text}]})
    chat = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return qwen_processor(text=[chat], images=[image], padding=False, return_tensors='pt').to(QWEN_DEVICE)

def load_rgb(path):
    with Image.open(path) as im:
        return ImageOps.exif_transpose(im).convert('RGB')

def word_spans(caption):
    return list(WORD_RE.finditer(caption))

def exact_caption20(raw):
    raw = raw.replace('\\n',' ').strip().strip('"“”\'')
    raw = re.sub(r'^(?:Caption\s*:\s*)', '', raw, flags=re.I)
    found = word_spans(raw)
    if len(found) < CAPTION_WORDS: return None, False, len(found)
    trunc = len(found)>CAPTION_WORDS
    result = raw[:found[CAPTION_WORDS-1].end()].strip(' ,;:')
    if result[-1] not in '.!?': result += '.'
    assert len(word_spans(result))==CAPTION_WORDS
    return result, trunc, len(found)

@torch.inference_mode()
def extract_caption20(image, reference_caption=None):
    attempts=[]
    for attempt in range(CAPTION_GENERATION_ATTEMPTS):
        msg = CAPTION_PROMPT if attempt==0 else (CAPTION_PROMPT +
            ' This is a strict word-count task: write at least '+ str(25+5*(attempt-1)) + ' concrete words in one sentence; never produce a short caption.')
        enc=qwen_inputs(image, msg)
        output=qwen.generate(**enc, max_new_tokens=MAX_QWEN_TOKENS, do_sample=False,
                             use_cache=True, pad_token_id=qwen_processor.tokenizer.pad_token_id)
        raw=qwen_processor.batch_decode(output[:,enc['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()
        caption,truncated,source_n=exact_caption20(raw)
        attempts.append({'attempt':attempt+1,'raw':raw,'word_count':source_n,'truncated':truncated})
        if caption:
            return caption, {'caption_origin':'qwen_generated','word_count':CAPTION_WORDS,
                             'generation_attempts':attempts,'reference_caption':reference_caption}
    raise ValueError('Qwen did not produce at least 20 words after retry: '+repr(attempts))

# POS/dependency selection, not lexical prompt engineering or a generic stopword blacklist.
ALLOWED_POS = {'NOUN','PROPN','VERB','ADJ','NUM'}
def select_content_tokens(caption, limit=MAX_CONTENT_TOKENS):
    doc=nlp(caption)
    ranked=[]
    for t in doc:
        if t.pos_ not in ALLOWED_POS or t.is_punct or t.is_space or not t.text.strip(): continue
        if t.pos_=='VERB' and t.dep_ in {'aux','auxpass'}: continue
        # Prefer heads/subjects and concrete modifiers, but retain original surface spelling & offsets.
        priority={'nsubj':0,'nsubjpass':0,'ROOT':1,'dobj':1,'obj':1,'pobj':2,
                  'compound':2,'amod':2,'acomp':3,'attr':3}.get(t.dep_,4)
        ranked.append({'token':t.text,'lemma':t.lemma_,'pos':t.pos_,'dep':t.dep_,
                       'start_char':int(t.idx),'end_char':int(t.idx+len(t)),
                       'priority':priority, 'token_index':int(t.i)})
    ranked.sort(key=lambda z:(z['priority'],z['token_index']))
    chosen=sorted(ranked[:limit],key=lambda z:z['token_index'])
    chunks=list(doc.noun_chunks)
    subjects=[t for t in doc if t.dep_ in {'nsubj','nsubjpass','csubj','csubjpass'} and t.pos_ in {'NOUN','PROPN'}]
    head=subjects[0] if subjects else next((t for t in doc if t.pos_ in {'NOUN','PROPN'}),None)
    if head is None: raise ValueError('Caption contains no grammatical noun to use as a subject: '+caption)
    chunk=next((c for c in chunks if c.start<=head.i<c.end),None)
    text=(chunk.text if chunk else head.text)
    subject_tokens=[t.text for t in nlp(text) if t.pos_ not in {'DET','PRON','PUNCT','ADP'}]
    subject=' '.join(subject_tokens).strip() or head.text
    return subject,chosen,ranked

## 4. Qwen attention for **full caption** and **each filtered content word**
The mapped attention comes from *query prompt tokens*, not decoder-generated words; the input image positions are checked against `image_grid_thw`. Query-based comparisons reuse the same prefix. Keep image-token probability mass and spatial maps separate.

In [ ]:
from transformers.models.qwen3_vl.modeling_qwen3_vl import apply_rotary_pos_emb
from contextlib import contextmanager

QWEN_NLAYERS = len(qwen.model.language_model.layers)
QWEN_LAYERS = sorted(set(min(QWEN_NLAYERS - 1, max(0, round((QWEN_NLAYERS - 1) * f)))
                         for f in QWEN_LAYER_FRACTIONS))
assert QWEN_LAYERS, 'No Qwen attention layers selected'
print('Selected Qwen language-model layers:', QWEN_LAYERS, '/', QWEN_NLAYERS)

# Minimal phrase query; ensures that the query phrase is at the end of user text.
QUERY_PREFIX = ('Inspect this image. Identify visual evidence for the following expression. '
                'Treat the expression as a query, not as an assertion of truth.\nEXPRESSION:\n')

def build_query_batch(image, suffix):
    # No system prompt is required for attention probing. Exactly one image per request.
    msgs = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text',
              'text': QUERY_PREFIX + suffix}]}]
    prompt = qwen_processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return qwen_processor(text=[prompt], images=[image], return_tensors='pt').to(QWEN_DEVICE)

def changed_span(base, full):
    a, b = base['input_ids'][0].tolist(), full['input_ids'][0].tolist()
    start = 0
    while start < min(len(a), len(b)) and a[start] == b[start]: start += 1
    end_a, end_b = len(a), len(b)
    while end_a > start and end_b > start and a[end_a - 1] == b[end_b - 1]:
        end_a -= 1; end_b -= 1
    return list(range(start, end_b))

@torch.inference_mode()
def qwen_phrase_attention(image, phrase, base=None):
    if base is None: base = build_query_batch(image, '')
    batch = build_query_batch(image, phrase)
    img_id = int(qwen.config.image_token_id)
    image_positions = torch.where(batch['input_ids'][0] == img_id)[0]
    if image_positions.numel() == 0:
        raise RuntimeError('No Qwen image tokens detected in input_ids')
    grid = batch['image_grid_thw'][0].tolist()
    merge = int(qwen.config.vision_config.spatial_merge_size)
    t, gh_raw, gw_raw = [int(x) for x in grid]
    assert gh_raw % merge == 0 and gw_raw % merge == 0
    gh, gw = gh_raw // merge, gw_raw // merge
    expected = t * gh * gw
    assert t == 1, f'Expected static image with one temporal grid, got {grid}'
    assert expected == image_positions.numel(), (
        f'Image-grid mismatch: {grid=} {merge=} {expected=} {image_positions.numel()=}')
    query_positions = changed_span(base, batch)
    assert query_positions and min(query_positions) > image_positions.max().item(), (
        'Could not align phrase tokens after image; examine tokenizer/template.')
    # Check that the differing prompt tokens actually represent the query phrase.
    found = qwen_processor.tokenizer.decode(batch['input_ids'][0, query_positions], skip_special_tokens=True)
    if re.sub(r'\s+', '', phrase).casefold() not in re.sub(r'\s+', '', found).casefold():
        raise ValueError(f'Unreliable phrase token alignment: target={phrase!r}, matched={found!r}')
    positions_t = torch.tensor(query_positions, device=QWEN_DEVICE, dtype=torch.long)
    img_positions_t = image_positions.to(QWEN_DEVICE)
    collected = {}
    handles = []

    def hook_for(index):
        def prehook(module, args, kwargs):
            hidden = kwargs.get('hidden_states', args[0] if args else None)
            cos, sin = kwargs['position_embeddings']
            shape = (*hidden.shape[:2], -1, module.head_dim)
            q = module.q_norm(module.q_proj(hidden).view(shape)).transpose(1, 2)
            k = module.k_norm(module.k_proj(hidden).view(shape)).transpose(1, 2)
            q, k = apply_rotary_pos_emb(q, k, cos, sin)
            k = k.repeat_interleave(module.num_key_value_groups, dim=1)
            # q: B,H,S,D ; k: B,H,S,D. Only compute requested query rows.
            q_part = q.index_select(-2, positions_t).float()
            logits = torch.matmul(q_part, k.float().transpose(-1, -2)) * module.scaling
            seq_idx = torch.arange(k.shape[-2], device=logits.device)
            causal_forbidden = seq_idx[None, :] > positions_t[:, None]
            logits.masked_fill_(causal_forbidden[None, None, :, :], torch.finfo(logits.dtype).min)
            probs = logits.softmax(-1)             # denominator includes all visible tokens
            image_probs = probs.index_select(-1, img_positions_t)  # B,H,Q,I
            token_maps = image_probs[0].mean(0)   # Q,I averaged across heads
            collected[index] = token_maps.cpu().numpy().astype('float32')
        return prehook

    try:
        for idx in QWEN_LAYERS:
            module = qwen.model.language_model.layers[idx].self_attn
            handles.append(module.register_forward_pre_hook(hook_for(idx), with_kwargs=True))
        _ = qwen(**batch, use_cache=False, output_attentions=False, return_dict=True)
    finally:
        for h in handles: h.remove()
    if set(collected) != set(QWEN_LAYERS):
        raise RuntimeError(f'Qwen hooks missing: {set(QWEN_LAYERS) - set(collected)}')
    per_token = np.stack([collected[i] for i in QWEN_LAYERS]).mean(0)  # Q,patches
    raw_mass = float(per_token.sum(-1).mean())
    raw_map = per_token.mean(0).reshape(gh, gw)
    tokens = [qwen_processor.tokenizer.decode([batch['input_ids'][0, p].item()],
                                               skip_special_tokens=False) for p in query_positions]
    return {'raw_map': raw_map, 'normalized_map': normalize_np(raw_map),
            'token_maps': per_token.reshape(len(tokens), gh, gw), 'tokens': tokens,
            'raw_image_mass': raw_mass, 'grid': [gh, gw], 'layer_ids': QWEN_LAYERS,
            'matched_query_tokens': found}

def normalize_np(arr):
    arr = np.nan_to_num(np.asarray(arr, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    amin, amax = float(arr.min()), float(arr.max())
    return (arr - amin) / (amax - amin + 1e-8)

## 5. Comparison methods on GPU 1
CLIP Surgery is **forward-based**, not gradient-based. Grad-ECLIP is the paper-specific gradient-weighted last-block method, not ordinary Grad-CAM. Both accept the exact same caption/content-word expressions as Qwen attention. CLIP Surgery and original CLIP use distinct forward architectures.

In [ ]:
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
clip_preprocess = Compose([
    Resize((CLIP_SIDE, CLIP_SIDE), interpolation=InterpolationMode.BICUBIC),
    ToTensor(), Normalize(CLIP_MEAN, CLIP_STD),
])

def torch_map(x, size):
    x = x.float().reshape(1, 1, *x.shape)
    return F.interpolate(x, size=size, mode='bilinear', align_corners=False)[0, 0].cpu().numpy()

@torch.inference_mode()
def clip_surgery_maps(image, phrases):
    x = clip_preprocess(image).unsqueeze(0).to(CLIP_DEVICE)
    feats = F.normalize(surgery_model.encode_image(x), dim=-1)
    assert feats.ndim == 3, f'Expect Surgery image patch features, got {feats.shape}'
    # Full CLIP text encoder, not prompt-ensemble: ensures phrases/caption are passed as written.
    tokens = surgery_clip.tokenize(phrases + [''], truncate=True).to(CLIP_DEVICE)
    txt = F.normalize(surgery_model.encode_text(tokens), dim=-1)
    maps = []
    for text_vec in txt[:-1]:
        sim = surgery_clip.clip_feature_surgery(feats, text_vec[None, :], redundant_feats=txt[-1:])
        scores = sim[0, 1:, 0]      # CLS token excluded
        side = math.isqrt(scores.numel())
        assert side * side == scores.numel(), 'Unexpected CLIP Surgery token spatial layout'
        maps.append(normalize_np(scores.reshape(side, side).float().cpu().numpy()))
    del x, feats, txt
    return maps

# Last block: one-head Q/K/V reconstruction from the Grad-ECLIP author notebook.
def grad_eclip_image_forward(x):
    v = clip_model.visual
    dtype = v.conv1.weight.dtype
    with torch.no_grad():
        patches = v.conv1(x.to(dtype))
        gh, gw = patches.shape[-2:]
        patches = patches.reshape(1, patches.shape[1], -1).permute(0, 2, 1)
        cls = v.class_embedding.to(dtype) + torch.zeros(1, 1, patches.shape[-1], device=CLIP_DEVICE, dtype=dtype)
        h = torch.cat([cls, patches], dim=1)
        # At CLIP_SIDE==224 exactly the official position embedding resolution.
        assert h.shape[1] == v.positional_embedding.shape[0], 'Set CLIP_SIDE=224 or add position interpolation.'
        h = v.ln_pre(h + v.positional_embedding.to(dtype))
        h = h.permute(1, 0, 2)  # S,B,C
        for blk in v.transformer.resblocks[:-1]:
            h = blk(h)
    h = h.detach().requires_grad_(True)
    blk = v.transformer.resblocks[-1]
    z = blk.ln_1(h)
    q, k, values = F.linear(z, blk.attn.in_proj_weight, blk.attn.in_proj_bias).chunk(3, dim=-1)
    # Single-head attention, as in author notebook's clip_encode_dense(..., n=1).
    q_scaled = q * (q.shape[-1] ** -0.5)
    probs = (q_scaled[:, 0] @ k[:, 0].T).softmax(dim=-1)  # S,S
    attn_output = (probs @ values[:, 0]).unsqueeze(1)    # S,1,C
    after_attn = F.linear(attn_output, blk.attn.out_proj.weight, blk.attn.out_proj.bias)
    h = h + after_attn
    h = h + blk.mlp(blk.ln_2(h))
    image_embedding = (v.ln_post(h.permute(1, 0, 2))[:, 0] @ v.proj)
    image_embedding = F.normalize(image_embedding.float(), dim=-1)
    return image_embedding, q, k, values, attn_output, (gh, gw)

@torch.no_grad()
def plain_clip_text(phrases):
    return F.normalize(clip_model.encode_text(openai_clip.tokenize(phrases, truncate=True).to(CLIP_DEVICE)).float(), dim=-1)

@torch.no_grad()
def plain_clip_score(image, phrases):
    emb = F.normalize(clip_model.encode_image(clip_preprocess(image).unsqueeze(0).to(CLIP_DEVICE)).float(), dim=-1)
    return (emb @ plain_clip_text(phrases).T)[0].float().cpu().numpy()

def grad_eclip_maps(image, phrases):
    xx = clip_preprocess(image).unsqueeze(0).to(CLIP_DEVICE)
    with torch.no_grad(): txt = plain_clip_text(phrases)
    emb, q, k, values, attn_output, grid = grad_eclip_image_forward(xx)
    scores = (emb @ txt.T).reshape(-1)
    with torch.no_grad():
        q_cls = F.normalize(q[0, 0].float(), dim=-1)
        k_patch = F.normalize(k[1:, 0].float(), dim=-1)
        relaxed = k_patch @ q_cls
        relaxed = (relaxed - relaxed.min()) / (relaxed.max() - relaxed.min() + 1e-8)
        val_patch = values[1:, 0].float()
    maps = []
    try:
        for j in range(len(phrases)):
            grad = torch.autograd.grad(scores[j], attn_output, retain_graph=j < len(phrases) - 1)[0]
            chan_weight = grad[0, 0].float()  # gradient at CLS attention output
            emap = (val_patch * chan_weight[None, :] * relaxed[:, None]).sum(-1).relu()
            maps.append(normalize_np(emap.reshape(grid).detach().cpu().numpy()))
        sims = scores.detach().cpu().numpy().astype('float32')
    finally:
        del emb, q, k, values, attn_output, xx
    return maps, sims

# Sanity check math without loading any data: Grad-ECLIP should differentiate a scalar
# w.r.t. the last attention output, not a detached tensor or a Surgery embedding.

## 6. Heatmap visualization and CLIP scoring utilities
A 7×7 **non-overlapping grid** is systematically blurred, evaluating the decrease in normal CLIP image–text cosine for *full caption* and *grammatical subject*. This is a contiguous intervention, not a pixelwise causal ground truth.

In [ ]:
def remove_phrase(caption, phrase):
    # Conservative literal substring deletion. If absent, the LOO estimate is undefined.
    match = re.search(re.escape(phrase), caption, flags=re.I)
    if not match: return None
    new = (caption[:match.start()] + caption[match.end():]).strip()
    new = re.sub(r'\s+', ' ', new)
    new = re.sub(r'\s+([,.!?])', r'\1', new)
    return new if new and new != caption else None

def resize_heatmap(m, wh):
    w, h = wh
    return cv2.resize(np.asarray(m, dtype='float32'), (w, h), interpolation=cv2.INTER_LINEAR)

def overlay(image, m, alpha=0.48):
    source = np.asarray(image.convert('RGB'), dtype=np.uint8)
    smooth = resize_heatmap(normalize_np(m), image.size)
    cmap = cv2.applyColorMap((smooth * 255).astype(np.uint8), cv2.COLORMAP_TURBO)
    cmap = cv2.cvtColor(cmap, cv2.COLOR_BGR2RGB)
    return np.clip(source * (1 - alpha) + cmap * alpha, 0, 255).astype(np.uint8)

def plot_panels(image, phrases, maps_by_method, image_id, out_dir, filename='comparison.png', row_labels=None):
    methods = ['Qwen3-VL attention', 'CLIP Surgery', 'Grad-ECLIP']
    nrows = len(phrases)
    fig, axes = plt.subplots(nrows, 4, figsize=(16, max(3, 2.9*nrows)), squeeze=False)
    for i, phrase in enumerate(phrases):
        axes[i, 0].imshow(image); axes[i, 0].set_title('Original / ' + (row_labels[i] if row_labels else ('FULL CAPTION' if i == 0 else 'EXPRESSION'))[:35], fontsize=10)
        axes[i, 0].set_ylabel((row_labels[i] if row_labels else phrase)[:65], fontsize=9)
        for j, method in enumerate(methods):
            axes[i, j+1].imshow(overlay(image, maps_by_method[method][i]))
            axes[i, j+1].set_title(method, fontsize=10)
        for a in axes[i]: a.axis('off')
    fig.suptitle(f'Image {image_id} — matched expressions / non-comparable map scales', fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    fname = out_dir / filename
    fig.savefig(fname, dpi=145, bbox_inches='tight')
    plt.show(); plt.close(fig)
    return fname

def plot_caption_tokens(image, attention_result, path):
    n = len(attention_result['tokens'])
    if n < 1: return
    cols = min(5, n)
    rows = math.ceil(n / cols)
    fig, ax = plt.subplots(rows, cols, figsize=(3.0 * cols, 2.7 * rows), squeeze=False)
    for j, a in enumerate(ax.flat):
        a.axis('off')
        if j < n:
            a.imshow(overlay(image, attention_result['token_maps'][j]))
            a.set_title(f'Qwen token {j}: {attention_result["tokens"][j]!r}', fontsize=9)
    fig.suptitle('Full-caption *prompt-token* attention (not token generation attribution)')
    fig.tight_layout(rect=[0, 0, 1, 0.96]); fig.savefig(path, dpi=125, bbox_inches='tight')
    plt.show(); plt.close(fig)

def gaussian_delete(image, heatmap, fraction=TOP_MASK_FRACTION, seed=SEED):
    src = np.asarray(image.convert('RGB'), dtype=np.uint8)
    H, W = src.shape[:2]
    hmap = resize_heatmap(normalize_np(heatmap), (W, H))
    k = max(1, min(H * W, int(fraction * H * W)))
    top_idx = np.argpartition(hmap.ravel(), -k)[-k:]
    mask = np.zeros(H*W, bool); mask[top_idx] = True
    # Fixed blur kernel for consistency on target and random control.
    blurred = cv2.GaussianBlur(src, (31, 31), 11.0)
    deleted = src.copy(); deleted.reshape(-1, 3)[mask] = blurred.reshape(-1, 3)[mask]
    rng = np.random.default_rng(seed)
    ridx = rng.choice(H*W, size=k, replace=False)
    random_deleted = src.copy()
    random_deleted.reshape(-1, 3)[ridx] = blurred.reshape(-1, 3)[ridx]
    return Image.fromarray(deleted), Image.fromarray(random_deleted)

# GPU 1 worker: single thread preserves CUDA device affinity; no Qwen access here.
def compute_clip_job(image, caption, concepts, image_id):
    torch.cuda.set_device(CLIP_DEVICE)
    phrases = [caption] + concepts
    surgery_maps = clip_surgery_maps(image, phrases)
    grad_maps, grad_scores = grad_eclip_maps(image, phrases)
    with torch.no_grad():
        orig_scores = plain_clip_score(image, phrases)
    assert len(surgery_maps) == len(grad_maps) == len(phrases)
    return {'surgery_maps': surgery_maps, 'grad_maps': grad_maps,
            'grad_scores': grad_scores.tolist(), 'plain_scores': orig_scores.tolist()}

def plot_impact_dashboard(image,clip_patch,q_patch,clip_words,qwen_words,image_id,out_dir):
    fig,axes=plt.subplots(2,3,figsize=(16,9))
    axes[0,0].imshow(image);axes[0,0].set_title('Original')
    for j,target in enumerate(('caption','subject')):
        df=clip_patch.query('target == @target').sort_values('patch_id')
        grid=np.full((PATCH_GRID,PATCH_GRID),np.nan,dtype=float)
        for r in df.itertuples(): grid[int(r.row),int(r.col)]=r.delta_clip
        # DO NOT minmax scale signed effect: preserve sign via symmetric axes.
        scale=max(float(np.max(np.abs(grid))),1e-6)
        im=axes[0,j+1].imshow(grid,cmap='coolwarm',vmin=-scale,vmax=scale)
        axes[0,j+1].set_title('CLIP patch delta: '+target)
        fig.colorbar(im,ax=axes[0,j+1],fraction=0.046)
        axes[0,j+1].set_xticks(range(PATCH_GRID));axes[0,j+1].set_yticks(range(PATCH_GRID))
    words=[f"{x['token']} ({x['pos']})" for x in clip_words]
    axes[1,0].barh(words,[x['clip_LOO_delta'] for x in clip_words]);axes[1,0].axvline(0,color='black',linewidth=0.8)
    axes[1,0].set_title('CLIP full-caption LOO \u0394 cosine');axes[1,0].invert_yaxis()
    axes[1,1].barh(words,[x['qwen_visual_logp_delta_sum'] for x in qwen_words]);axes[1,1].axvline(0,color='black',linewidth=0.8)
    axes[1,1].set_title('Qwen word visual \u0394 logp');axes[1,1].invert_yaxis()
    axes[1,2].imshow(image)
    for ax in axes.flat: ax.tick_params(labelsize=7)
    if len(q_patch):
        for r in q_patch.query("target == 'caption'").itertuples():
            a,b,c,d=patch_bbox(image.size,int(r.row),int(r.col))
            import matplotlib.patches as patches
            axes[1,2].add_patch(patches.Rectangle((a,b),c-a,d-b,fill=False,linewidth=2,
                                                   edgecolor='red' if r.delta_avg_logp>0 else 'blue'))
            axes[1,2].text(a,b,str(int(r.patch_id)),color='white',fontsize=8,
                           bbox={'facecolor':'black','alpha':0.7,'pad':0.5})
    axes[1,2].set_title('Qwen tested patches (sampled only)')
    axes[1,2].axis('off');axes[0,0].axis('off')
    fig.suptitle('Prediction sensitivity diagnostics | '+str(image_id),fontsize=13)
    fig.tight_layout()
    fname=out_dir/'importance_dashboard.png'
    fig.savefig(fname,dpi=145,bbox_inches='tight');plt.show();plt.close(fig)
    return fname

## 6b. Importance ranking and Qwen teacher-forcing likelihood
**Qwen score:** mean log-probability per target subword for a **fixed subject phrase / caption**, evaluated with the same prompt before and after blur. `Δ logp = logp(original)-logp(ablated)`. Positive means the untouched image supports that fixed string more. Per-caption-token `logp(image)-logp(fully_blurred_image)` is a **visual dependence** score (not free-generation importance).

**CLIP score:** cosine of full caption / subject before and after deleting a word or grid patch. `score(full)-score(without_word)` can be negative. Words and Qwen BPE tokens are aligned only through tokenizer offsets; their definitions are not interchangeable.

In [ ]:
def clip_word_loo(image, caption, selected, image_id, out_dir):
    baseline=float(plain_clip_score(image,[caption])[0])
    drops=[]
    for item in selected:
        a,b=item['start_char'],item['end_char']
        rewrite=re.sub(r'\s+',' ',(caption[:a]+caption[b:]).strip())
        rewrite=re.sub(r'\s+([,.!?])',r'\1',rewrite)
        drops.append(rewrite)
    with torch.no_grad():
        imgfeat=F.normalize(clip_model.encode_image(clip_preprocess(image).unsqueeze(0).to(CLIP_DEVICE)).float(),dim=-1)
        txt=plain_clip_text(drops)
        vals=(imgfeat@txt.T)[0].float().cpu().numpy()
    rows=[]
    for item,after,v in zip(selected,drops,vals):
        rows.append({**item,'image_id':image_id,'caption':caption,
                     'caption_without_word':after,'clip_original':baseline,'clip_without':float(v),
                     'clip_LOO_delta':baseline-float(v)})
    pd.DataFrame(rows).to_csv(out_dir/'selected_token_clip_loo.csv',index=False)
    return rows

def patch_bbox(wh, row,col, grid=PATCH_GRID):
    w,h=wh
    return (round(col*w/grid), round(row*h/grid),
            round((col+1)*w/grid), round((row+1)*h/grid))

def patch_blur(image,patch, grid=PATCH_GRID):
    src=np.asarray(image.convert('RGB')).copy()
    w,h=image.size
    row,col=divmod(int(patch),grid)
    x1,y1,x2,y2=patch_bbox((w,h),row,col,grid)
    kernel=min(PATCH_BLUR_KERNEL, max(3,(min(h,w)//2)*2-1))
    if kernel%2==0: kernel-=1
    blurred=cv2.GaussianBlur(src,(kernel,kernel),sigmaX=10.0)
    src[y1:y2,x1:x2]=blurred[y1:y2,x1:x2]
    return Image.fromarray(src)

@torch.inference_mode()
def clip_grid_occlusion(image,targets,image_id,out_dir,selected=None):
    # All PATCH_GRID**2 patches, evaluated under *vanilla* CLIP; batch images.
    selected=selected or []
    rewrites=[]
    caption=targets[0]
    for item in selected:
        a,b=item['start_char'],item['end_char']
        no_word=re.sub(r'\s+',' ',(caption[:a]+caption[b:]).strip())
        rewrites.append(re.sub(r'\s+([,.!?])',r'\1',no_word))
    tfeat=plain_clip_text(targets+rewrites)
    original=clip_preprocess(image).unsqueeze(0).to(CLIP_DEVICE)
    basefeat=F.normalize(clip_model.encode_image(original).float(),dim=-1)
    base=(basefeat@tfeat.T)[0].float().cpu().numpy()
    rows=[]; interactions=[]
    for first in range(0,PATCH_GRID**2,CLIP_PATCH_BATCH):
        idx=list(range(first,min(PATCH_GRID**2,first+CLIP_PATCH_BATCH)))
        ims=torch.stack([clip_preprocess(patch_blur(image,k)) for k in idx]).to(CLIP_DEVICE)
        emb=F.normalize(clip_model.encode_image(ims).float(),dim=-1)
        scores=(emb@tfeat.T).float().cpu().numpy()
        for i,k in enumerate(idx):
            rr,cc=divmod(k,PATCH_GRID)
            delta_full=float(base[0]-scores[i,0])
            for tindex,item in enumerate(selected):
                j=tindex+2
                delta_without=float(base[j]-scores[i,j])
                interactions.append({'image_id':image_id,'token':item['token'],
                                     'token_index':item['token_index'],'patch_id':k,
                                     'row':rr,'col':cc,'delta_clip_interaction':delta_full-delta_without})
            for j,name in enumerate(('caption','subject')):
                rows.append({'image_id':image_id,'patch_id':k,'row':rr,'col':cc,
                             'target':name,'base_clip':float(base[j]),
                             'masked_clip':float(scores[i,j]),
                             'delta_clip':float(base[j]-scores[i,j])})
    df=pd.DataFrame(rows)
    df.to_csv(out_dir/'clip_patch_occlusion_all.csv',index=False)
    pd.DataFrame(interactions).to_csv(out_dir/'clip_filtered_token_patch_interactions.csv',index=False)
    return df

# Teacher-forced, **same image/question and fixed string** for original vs ablated images.
@torch.inference_mode()
def qwen_target_logprobs(image,target,question):
    enc=qwen_inputs(image,question)
    tokenizer=qwen_processor.tokenizer
    enc_target=tokenizer(target,add_special_tokens=False,return_offsets_mapping=True)
    target_ids=torch.tensor(enc_target['input_ids'],device=QWEN_DEVICE,dtype=torch.long)
    if target_ids.numel()==0: raise ValueError('Empty Qwen score target')
    seq=torch.cat((enc['input_ids'],target_ids[None,:]),dim=1)
    # attention mask is only needed for simple unpadded batch, all ones.
    kwargs={k:v for k,v in enc.items() if k!='input_ids' and k!='attention_mask'}
    for key in ('token_type_ids','mm_token_type_ids'):
        if key in kwargs and kwargs[key].ndim==2 and kwargs[key].shape[-1]==plen:
            kwargs[key]=F.pad(kwargs[key],(0,len(target_ids)),value=0)
    logits=qwen(input_ids=seq,attention_mask=torch.ones_like(seq),use_cache=False,
                return_dict=True,**kwargs).logits
    plen=enc['input_ids'].shape[1]
    pred=logits[0,plen-1:plen+len(target_ids)-1].float()
    logps=pred.log_softmax(-1).gather(-1,target_ids[:,None]).squeeze(-1)
    vals=logps.detach().cpu().numpy()
    tokens=tokenizer.convert_ids_to_tokens(target_ids.cpu().tolist())
    offsets=[(int(a),int(b)) for a,b in enc_target['offset_mapping']]
    assert len(vals)==len(tokens)==len(offsets)
    return {'mean_logp':float(vals.mean()),'token_logps':vals,'tokens':tokens,'offsets':offsets}

def uniform_blur(image):
    src=np.asarray(image.convert('RGB'))
    h,w=src.shape[:2]
    kernel=min(61,max(3,(min(w,h)//2)*2-1))
    if kernel%2==0: kernel-=1
    return Image.fromarray(cv2.GaussianBlur(src,(kernel,kernel),sigmaX=22.0))

def calc_qwen_token_visual_dependence(image,caption,subject,selected,image_id,out_dir):
    complete=qwen_target_logprobs(image,caption,CAPTION_PROMPT)
    blank=qwen_target_logprobs(uniform_blur(image),caption,CAPTION_PROMPT)
    if complete['offsets']!=blank['offsets']: raise RuntimeError('Qwen target alignment changed')
    tok_rows=[]
    for i,(tok,(a,b),p,blank_p) in enumerate(zip(complete['tokens'],complete['offsets'],complete['token_logps'],blank['token_logps'])):
        tok_rows.append({'image_id':image_id,'bpe_index':i,'token':tok,
                         'start_char':a,'end_char':b,'logp_image':float(p),
                         'logp_blurred_image':float(blank_p),'visual_logp_delta':float(p-blank_p)})
    pd.DataFrame(tok_rows).to_csv(out_dir/'qwen_caption_generated_token_logp.csv',index=False)
    token_scores=[]
    for item in selected:
        a,b=item['start_char'],item['end_char']
        hit=[r for r in tok_rows if r['start_char']<b and r['end_char']>a]
        token_scores.append({**item,'image_id':image_id,'qwen_visual_logp_delta_sum':sum(r['visual_logp_delta'] for r in hit),
                             'qwen_bpe_indices':','.join(str(r['bpe_index']) for r in hit)})
    pd.DataFrame(token_scores).to_csv(out_dir/'filtered_token_qwen_visual_scores.csv',index=False)
    sub=qwen_target_logprobs(image,subject,SUBJECT_SCORE_PROMPT)
    sub_blur=qwen_target_logprobs(uniform_blur(image),subject,SUBJECT_SCORE_PROMPT)
    rows_subject=[]
    for i,(tok,offset,p,blank_p) in enumerate(zip(sub['tokens'],sub['offsets'],sub['token_logps'],sub_blur['token_logps'])):
        rows_subject.append({'image_id':image_id,'bpe_index':i,'token':tok,
                             'start_char':offset[0],'end_char':offset[1],
                             'logp_image':float(p),'logp_blurred_image':float(blank_p),
                             'visual_logp_delta':float(p-blank_p)})
    pd.DataFrame(rows_subject).to_csv(out_dir/'qwen_subject_generated_token_logp.csv',index=False)
    return {'caption':complete,'subject':sub,'filtered_rows':token_scores,
            'bpe_rows':tok_rows,'subject_bpe_rows':rows_subject}

def select_qwen_patches(maps_by_method,max_budget=QWEN_PATCH_BUDGET,seed=SEED):
    # Candidate union: best caption + best subject region for every heatmap method.
    candidates=[]
    for method,hmaps in maps_by_method.items():
        for ti in [0,1]:  # full caption, subject
            hm=resize_heatmap(hmaps[ti],(PATCH_GRID,PATCH_GRID))
            candidates.append((int(hm.reshape(-1).argmax()),f'{method}/{"caption" if ti==0 else "subject"}'))
    rng=np.random.default_rng(seed)
    candidates.append((int(rng.integers(PATCH_GRID**2)),'random_control'))
    selected=[]; sources={}
    for k,src in candidates:
        if k not in sources:
            selected.append(k);sources[k]=[]
        sources[k].append(src)
    return selected[:max_budget],sources

def qwen_patch_occlusion(image,caption,subject,originals,patch_ids,sources,image_id,out_dir):
    rows=[]; matrix=[]
    for k in patch_ids:
        im=patch_blur(image,k)
        m_caption=qwen_target_logprobs(im,caption,CAPTION_PROMPT)
        m_sub=qwen_target_logprobs(im,subject,SUBJECT_SCORE_PROMPT)
        if len(m_caption['token_logps'])!=len(originals['caption']['token_logps']):
            raise RuntimeError('Token IDs/target length changed on patch intervention')
        token_drop=originals['caption']['token_logps']-m_caption['token_logps']
        matrix.append(token_drop)
        for name,score in [('caption',m_caption),('subject',m_sub)]:
            base=originals[name]['mean_logp']
            rows.append({'image_id':image_id,'patch_id':k,'row':k//PATCH_GRID,'col':k%PATCH_GRID,
                         'selection_source':'|'.join(sources[k]),'target':name,
                         'base_avg_logp':base,'occluded_avg_logp':score['mean_logp'],
                         'delta_avg_logp':base-score['mean_logp']})
    df=pd.DataFrame(rows)
    df.to_csv(out_dir/'qwen_patch_occlusion_sampled.csv',index=False)
    if len(matrix):
        delta_matrix=np.stack(matrix)
        np.savez_compressed(out_dir/'qwen_patch_by_generated_token.npz',
            patch_ids=np.array(patch_ids,dtype=int), caption_bpe_patch_logp_delta=delta_matrix,
            bpe_tokens=np.asarray(originals['caption']['tokens'],dtype=str))
        interactions=[]
        for word in originals['filtered_rows']:
            ids=[int(t) for t in word['qwen_bpe_indices'].split(',') if t.strip()]
            for i,k in enumerate(patch_ids):
                interactions.append({'image_id':image_id,'token':word['token'],
                                     'token_index':word['token_index'],'patch_id':k,
                                     'delta_token_logp':float(delta_matrix[i,ids].sum()) if ids else float('nan')})
        pd.DataFrame(interactions).to_csv(out_dir/'qwen_filtered_token_patch_interactions.csv',index=False)
    return df

def plot_token_patch_interactions(image_id,folder):
    import matplotlib.pyplot as plt
    from matplotlib.colors import TwoSlopeNorm
    path=Path(folder)
    files=[('CLIP word × patch interaction',path/'clip_filtered_token_patch_interactions.csv','delta_clip_interaction'),
           ('Qwen word × patch (tested only)',path/'qwen_filtered_token_patch_interactions.csv','delta_token_logp')]
    fig,axs=plt.subplots(1,2,figsize=(18,5),squeeze=False)
    for a,(title,file,col) in zip(axs.flat,files):
        if not file.exists(): a.set_axis_off();continue
        df=pd.read_csv(file)
        if df.empty: a.set_axis_off();continue
        pvt=df.pivot_table(index=['token_index','token'],columns='patch_id',values=col,aggfunc='mean')
        labels=[str(i[1])+' ['+str(i[0])+']' for i in pvt.index]
        values=pvt.to_numpy(dtype=float)
        v=max(float(np.nanmax(np.abs(values))),1e-9)
        im=a.imshow(values,aspect='auto',cmap='coolwarm',vmin=-v,vmax=v)
        a.set_yticks(range(len(labels)),labels)
        a.set_xticks(range(len(pvt.columns)),[str(k) for k in pvt.columns],fontsize=6,rotation=90)
        a.set_xlabel('Patch ID (7×7 row-major order)')
        a.set_title(title)
        fig.colorbar(im,ax=a,fraction=0.04)
    fig.suptitle(str(image_id)+' — joint word/patch sensitivity (signed score changes)')
    fig.tight_layout()
    fig.savefig(path/'filtered_token_patch_interactions.png',dpi=145,bbox_inches='tight')
    plt.show();plt.close(fig)


## 7. Two-GPU experiment: live original images, paired full-caption/subject/selected-token maps, intervention scores
The original image and 20-word caption are displayed before the maps. Both CLIP architectures run on GPU1 while Qwen runs on GPU0. GPU1 work is placed in a bounded thread executor; Qwen baseline likelihood computation can overlap it. CLIP patch grid is exhaustively evaluated (49 regions); Qwen patch test is a *sampled subset*. **No claims of full-grid Qwen ranking** outside tested patches.

In [ ]:
from tqdm.auto import tqdm
results=[]; manifest=[]
executor=ThreadPoolExecutor(max_workers=1,thread_name_prefix='clip_gpu1')

def compute_clip_all(image,phrases,caption,subject,image_id,out_dir):
    torch.cuda.set_device(CLIP_DEVICE)
    surgery=clip_surgery_maps(image,phrases)
    grade,scores=grad_eclip_maps(image,phrases)
    original=plain_clip_score(image,phrases).tolist()
    clip_patch=clip_grid_occlusion(image,[caption,subject],image_id,out_dir,selected_words_by_image[image_id])
    selected_loo=clip_word_loo(image,caption,selected_words_by_image[image_id],image_id,out_dir)
    return {'surgery':surgery,'grade':grade,'reconstructed_scores':scores.tolist(),
            'clip_scores':original,'clip_patch':clip_patch,'selected_loo':selected_loo}

selected_words_by_image={}
try:
    for sample_index,sample in enumerate(tqdm(samples,desc='Image analysis')):
        image=load_rgb(sample['image_path'])
        image_id=str(sample['image_id'])
        safe=re.sub(r'[^a-zA-Z0-9_-]','_',image_id)[:70]
        out_dir=OUTPUT_DIR/f'{sample_index:04d}_{safe}'
        out_dir.mkdir(exist_ok=True,parents=True)
        image.save(out_dir/'original.png')
        caption,caption_meta=extract_caption20(image,sample.get('reference_caption'))
        subject,selected,all_pos_candidates=select_content_tokens(caption)
        selected_words_by_image[image_id]=selected
        assert len(word_spans(caption))==20,caption
        # Show the actual source image, generated caption and human-readable POS rationale.
        print('\nIMAGE:',image_id,'| CAPTION [20 words]:',caption,'| SUBJECT:',subject)
        display(image.resize((min(640,image.width),int(image.height*min(640,image.width)/image.width))))
        display(pd.DataFrame(selected)[['token','pos','dep','start_char','end_char']])
        phrases=[caption,subject]+[x['token'] for x in selected]
        labels=['FULL CAPTION','SUBJECT']+[f"{x['token']} [{x['pos']}]" for x in selected]
        base=build_query_batch(image,'')
        q_records=[]; q_maps=[]
        for expression in phrases:
            entry=qwen_phrase_attention(image,expression,base=base)
            q_maps.append(entry['normalized_map'])
            q_records.append({k:v for k,v in entry.items() if k not in ('normalized_map','raw_map','token_maps')})
        # Launch GPU1 work; process Qwen likelihood on GPU0 concurrently.
        clip_future=executor.submit(compute_clip_all,image,phrases,caption,subject,image_id,out_dir)
        q_scores=calc_qwen_token_visual_dependence(image,caption,subject,selected,image_id,out_dir)
        clip_out=clip_future.result()
        maps={'Qwen3-VL attention':q_maps, 'CLIP Surgery':clip_out['surgery'],
              'Grad-ECLIP':clip_out['grade']}
        # Both panels are displayed inline AND saved to disk.
        plot_panels(image,phrases[:1],{k:v[:1] for k,v in maps.items()},image_id,
                    out_dir,filename='full_caption_comparison.png',row_labels=labels[:1])
        plot_panels(image,phrases[1:2],{k:v[1:2] for k,v in maps.items()},image_id,
                    out_dir,filename='subject_comparison.png',row_labels=labels[1:2])
        plot_panels(image,phrases[2:],{k:v[2:] for k,v in maps.items()},image_id,
                    out_dir,filename='selected_tokens_comparison.png',row_labels=labels[2:])
        # Complete CLIP patches; Qwen patches selected from both caption/subject maps.
        patch_ids,patch_sources=select_qwen_patches(maps,seed=SEED+sample_index)
        q_patch=qwen_patch_occlusion(image,caption,subject,q_scores,patch_ids,patch_sources,image_id,out_dir) if RUN_QWEN_PATCH_TEST else pd.DataFrame()
        plot_impact_dashboard(image,clip_out['clip_patch'],q_patch,
                              clip_out['selected_loo'],q_scores['filtered_rows'],image_id,out_dir)
        plot_token_patch_interactions(image_id,out_dir)
        if SAVE_NPZ:
            arrays={}
            for method,hs in maps.items():
                tag={'Qwen3-VL attention':'qwen_attention','CLIP Surgery':'clip_surgery','Grad-ECLIP':'grad_eclip'}[method]
                for i,hm in enumerate(hs):arrays[f'{tag}_{i}']=np.asarray(hm,dtype=np.float32)
            np.savez_compressed(out_dir/'maps.npz',**arrays)
        for j,(name,expression) in enumerate(zip(labels,phrases)):
            results.append({'image_id':image_id,'expression':expression,'kind':name,
                            'caption':caption,'grammatical_subject':subject,
                            'qwen_image_attention_mass':q_records[j]['raw_image_mass'],
                            'clip_similarity':clip_out['clip_scores'][j],
                            'grad_eclip_reconstructed_similarity':clip_out['reconstructed_scores'][j]})
        # Save metadata with explicit score semantics and inspected subset.
        meta={'image_id':image_id,'image_path':sample['image_path'],
              'reference_caption':sample.get('reference_caption'), 'caption':caption,
              'caption_metadata':caption_meta,'caption_word_count':len(word_spans(caption)),
              'subject':subject,'selected_content_tokens':selected,
              'all_content_candidates':all_pos_candidates,'selected_expressions':phrases,
              'qwen_attention_records':q_records,'qwen_patch_ids_tested':patch_ids,
              'qwen_patch_sources':patch_sources,'patch_grid':PATCH_GRID,
              'qwen_patch_grid_exhaustive':False,
              'scoring_note':'Qwen: fixed target teacher-forced avg logp. CLIP: cosine. Masked patches are blurred.'}
        (out_dir/'metadata.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False,default=str))
        manifest.append({'image_id':image_id,'folder':str(out_dir),'caption':caption,'subject':subject,
                         'selected_tokens':[t['token'] for t in selected]})
        pd.DataFrame(results).to_csv(OUTPUT_DIR/'expression_summary.csv',index=False)
        (OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
        print('Exported:',out_dir)
finally:
    executor.shutdown(wait=True)
print('Completed images:',len(manifest),'| Results:',OUTPUT_DIR)

## 8. Inspect quantitative output across images
`selected_token_clip_loo.csv`: leave-one-word-out effect on CLIP full-caption cosine; `filtered_token_qwen_visual_scores.csv`: mean/summed generated-token logp image-versus-fully-blurred dependence; `clip_patch_occlusion_all.csv`: full 49-grid for caption and subject; `qwen_patch_occlusion_sampled.csv`: **only selected regions** and scores for subject/caption. `filtered_token_patch_interactions.png`: CLIP word×patch interaction and selected Qwen word×patch likelihood drop. Read signed effects, and inspect negative deltas; compare evidence under each evaluator separately. No ground-truth localization metrics are implied.

In [ ]:
summary=pd.read_csv(OUTPUT_DIR/'expression_summary.csv')
display(summary.head(24))
for info in manifest:
    d=Path(info['folder'])
    print('\nImage:',info['image_id'],'|',info['caption'])
    cw=pd.read_csv(d/'selected_token_clip_loo.csv')
    qw=pd.read_csv(d/'filtered_token_qwen_visual_scores.csv')
    display(cw[['token','pos','dep','clip_LOO_delta']].sort_values('clip_LOO_delta',ascending=False))
    display(qw[['token','pos','qwen_visual_logp_delta_sum']].sort_values('qwen_visual_logp_delta_sum',ascending=False))
    print('Qwen subject BPE image-dependence (fixed subject target):')
    display(pd.read_csv(d/'qwen_subject_generated_token_logp.csv')[['token','visual_logp_delta']].sort_values('visual_logp_delta',ascending=False))
    cp=pd.read_csv(d/'clip_patch_occlusion_all.csv')
    for target in ['caption','subject']:
        top=cp.loc[cp.target==target].sort_values('delta_clip',ascending=False).head(5)
        print('Top five CLIP patch drops for',target)
        display(top[['patch_id','row','col','delta_clip']])
    qp=d/'qwen_patch_occlusion_sampled.csv'
    if qp.exists():
        print('Qwen: only sampled patches tested (not exhaustive)')
        display(pd.read_csv(qp).sort_values('delta_avg_logp',ascending=False).head(10))

## 9. Correct scientific interpretation and next ablations

- **Top CLIP word** = highest signed `clip_LOO_delta` among selected caption words, **not** proof of the decisive word for Qwen generation. Also examine absolute magnitude (sign matters).
- **Top Qwen word** = highest `logp(image)-logp(fully blurred image)` over its overlapping BPE target tokens; this measures sensitivity to an **entire blurred image**, not standalone word importance. Compare with token logp/perplexity.
- **Top CLIP patch** = strongest loss of CLIP cosine after blur of one of the **49 equal-area grid regions**, separately for full caption and parsed subject.
- **Top Qwen patch** = strongest reduction in fixed-target Qwen mean logp **among the recorded selected patches only**, not all 49. Selection uses peaks from saliency maps plus random control and is therefore biased; change `QWEN_PATCH_BUDGET` or use a full grid for exhaustive Qwen attribution.
- **Attention/Grad-ECLIP vs actual influence:** look at map peak overlap with high-delta patches and observe disagreement; all models use different image preprocessing and might localize differently. The Qwen visual/text score is not equal to vanilla CLIP cosine.
- Captions are 20 **regex words**, Qwen uses subword tokens. Words truncated to a length limit can end ungrammatically; `caption_metadata` records truncation. A grammatical subject need not be the dominant visible object.
- For quantitative claims beyond exploratory work, collect human object masks, report stability over masking baselines (blur vs constant vs inpainting), bootstrap CIs and repeated samples, and perform a balanced dataset-wide aggregation.